In [2]:
import numpy as np
import pandas as pd
import difflib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import joblib

In [2]:
data = pd.read_csv("books_with_emotions.csv")

In [3]:
data

,Title,description,authors,categories,image,previewLink,unique_values,tagged_description,simple_categories,anger,disgust,fear,joy,sadness,surprise,neutral
0,Dr. Seuss: American Icon,Philip Nel takes a fascinating look into the k...,Philip Nel,Biography & Autobiography,http://books.google.com/books/content?id=IjvHQ...,http://books.google.nl/books?id=IjvHQsCn_pgC&p...,1,1 Philip Nel takes a fascinating look into the...,Nonfiction,0.034711,0.806784,0.082411,0.245415,0.825417,0.029443,0.264740
1,Wonderful Worship in Smaller Churches,This resource includes twelve principles in un...,David R. Ray,Religion,http://books.google.com/books/content?id=2tsDA...,http://books.google.nl/books?id=2tsDAAAACAAJ&d...,2,2 This resource includes twelve principles in ...,Nonfiction,0.064134,0.104007,0.051363,0.040564,0.964106,0.111690,0.078765
2,Whispers of the Wicked Saints,Julia Thomas finds her life spinning out of co...,Veronica Haddon,Fiction,http://books.google.com/books/content?id=aRSIg...,http://books.google.nl/books?id=aRSIgJlq6JwC&d...,3,3 Julia Thomas finds her life spinning out of ...,Fiction,0.974435,0.913045,0.351812,0.212945,0.549477,0.727175,0.185956
3,The Church of Christ: A Biblical Ecclesiology ...,In The Church of Christ: A Biblical Ecclesiolo...,Everett Ferguson,Religion,http://books.google.com/books/content?id=kVqRa...,http://books.google.nl/books?id=kVqRaiPlx88C&p...,4,4 In The Church of Christ: A Biblical Ecclesio...,Nonfiction,0.064134,0.104007,0.051363,0.040564,0.934228,0.111690,0.078765
4,Saint Hyacinth of Poland,The story for children 10 and up of St. Hyacin...,Mary Fabyan Windeatt,Biography & Autobiography,http://books.google.com/books/content?id=lmLqA...,http://books.google.nl/books?id=lmLqAAAACAAJ&d...,5,5 The story for children 10 and up of St. Hyac...,Nonfiction,0.053465,0.791916,0.059444,0.346165,0.878395,0.055465,0.156406
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109783,Final things,Grace's father believes in science and builds ...,Jenny Offill,Fiction,http://books.google.com/books/content?id=UbSFB...,http://books.google.com/books?id=UbSFBAAAQBAJ&...,128104,128104 Grace's father believes in science and ...,Fiction,0.064134,0.901087,0.959156,0.849096,0.785265,0.455001,0.340976
109784,The Magic of the Soul: Applying Spiritual Powe...,"""The Magic of the Soul, Applying Spiritual Pow...",Patrick J. Harbula,"Body, Mind & Spirit",http://books.google.com/books/content?id=H1ELA...,http://books.google.com/books?id=H1ELAAAACAAJ&...,128105,"128105 ""The Magic of the Soul, Applying Spirit...",NaN,0.064134,0.104007,0.051363,0.887439,0.938217,0.111690,0.078765
109785,Autodesk Inventor 10 Essentials Plus,Autodesk Inventor 2017 Essentials Plus provide...,"Daniel Banach, Travis Jones",Computers,http://books.google.com/books/content?id=zxHRC...,http://books.google.com/books?id=zxHRCwAAQBAJ&...,128106,128106 Autodesk Inventor 2017 Essentials Plus ...,Fiction,0.064134,0.104007,0.051363,0.188964,0.967443,0.111690,0.078765
109786,The Orphan Of Ellis Island (Time Travel Advent...,"During a school trip to Ellis Island, Dominick...",Elvira Woodruff,Juvenile Fiction,http://books.google.com/books/content?id=J7M-N...,http://books.google.com/books?id=J7M-NwAACAAJ&...,128107,"128107 During a school trip to Ellis Island, D...",Fiction,0.064134,0.104007,0.065212,0.248225,0.577747,0.111690,0.078765


In [5]:
titles = data["Title"].fillna("").tolist()

tfidf = TfidfVectorizer(stop_words="english")

tfidf_matrix = tfidf.fit_transform(titles)

In [7]:
joblib.dump(tfidf, "tfidf_vectorizer.pkl")
joblib.dump(tfidf_matrix, "tfidf_matrix.pkl")
joblib.dump(data, "books_df.pkl")   

print("TF-IDF saved successfully!")

TF-IDF saved successfully!


In [3]:
tfidf = joblib.load("tfidf_vectorizer.pkl")
tfidf_matrix = joblib.load("tfidf_matrix.pkl")
books = joblib.load("books_df.pkl")

In [4]:
def recommend_titles(query: str, top_k: int = 5):
    query_vec = tfidf.transform([query])

    scores = cosine_similarity(query_vec, tfidf_matrix)[0]

    best_idx = scores.argsort()[-top_k:][::-1]

    results = []
    for idx in best_idx:
        results.append({
            "title": books.iloc[idx]["Title"],
            "similarity": float(scores[idx])
        })
    return results

In [6]:
results = recommend_titles("dragon chronicles", top_k=10)
for r in results:
    print(r)

{'title': 'Dragon Chronicles', 'similarity': 1.0000000000000002}
{'title': "Shadow of the Dragon: Dragon's Fire (Book 2)", 'similarity': 0.6396622046550338}
{'title': 'The dragon and the book,', 'similarity': 0.6390799045327872}
{'title': 'Dragon World', 'similarity': 0.6221135024186296}
{'title': 'Dragon America (v. 2)', 'similarity': 0.6027736598807147}
{'title': 'The dragon in the sea', 'similarity': 0.5494766814830503}
{'title': "Dragon's Blood", 'similarity': 0.5392914084155347}
{'title': 'The Paper Dragon', 'similarity': 0.5239504654576983}
{'title': 'The dragon stone', 'similarity': 0.5216216004599878}
{'title': "The Dragon's Eye", 'similarity': 0.5214022626185703}
